# 5. A photoheated Strömgren sphere with a stellar wind

This is an advanced example combining spherical hydrodynamics, hydrogen chemistry, radiative transfer, photoheating, and a stellar-wind inner boundary. The notebook uses RadHydropy's validated C²-Ray smoke configuration: a 20 pc cloud, 64 hydro cells, a short 0.05 Myr evolution, and three saved output times.

The goal is to understand how two different sources of feedback interact: ionizing photons create a hot H II region, while the wind injects mass and momentum and drives a shocked bubble.

## Physics of the problem

A central source emits ionizing photons at rate `N_gamma = 10^49 s^-1` into initially neutral hydrogen with number density `n_H = 100 cm^-3` and temperature `100 K`. The radiation field photoionizes hydrogen and deposits excess photon energy as heat. Hydrogen recombination competes with photoionization, so the ionization front is determined by both photon supply and the evolving gas density.

For a uniform, static cloud, the classical Strömgren radius is

$$R_S=\left(\frac{3\dot N_\gamma}{4\pi\alpha_B n_H^2}\right)^{1/3},$$

where `alpha_B` is the case-B recombination coefficient. The actual front can differ from this estimate because the gas is heated and compressed, the wind changes the density profile, and the simulation is time-dependent.

The wind has mass-loss rate `dot M = 10^-6 M_sun yr^-1`, velocity `1000 km s^-1`, and injection radius `0.05 pc`. Its ram pressure at radius `r` is approximately

$$P_{\rm wind}(r)=\frac{\dot M v_{\rm wind}}{4\pi r^2}.$$

The wind shocks against the photoheated ambient gas. The resulting shell, hot wind cavity, ionization front, and photoheated region evolve together.

## Equations and numerical operators

The gas follows the spherical Euler equations, augmented by source terms for gravity-free hydrogen chemistry, radiative transfer, photoheating, and the wind boundary. In radial symmetry, mass, radial momentum, and total energy are transported through spherical shell faces. The hydrogen state includes the neutral fraction `x_HI`; its evolution schematically has the form

$$\frac{d x_{\rm HI}}{dt} = -\Gamma_{\rm ph}x_{\rm HI} + \alpha_B(T)n_e x_{\rm HII} + \text{collisional terms},$$

where `Gamma_ph` is the local photoionization rate. Photoheating adds energy according to the absorbed ionizing photon rate and the excess energy above the ionization threshold. In this configuration collisional ionization is disabled, while recombination and photoheating are active.

The long-characteristics radiative-transfer solver transports ionizing photons from the centre through the shells. The C²-Ray temporal scheme iterates the radiation and chemistry state within each source update. The split-implicit hydrogen solver then handles stiff chemistry and thermal coupling without requiring the global hydrodynamic timestep to resolve every reaction timescale.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO = Path('/content/RadHydropy')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/tkc004/RadHydropy.git', str(REPO)], check=True)
PINNED = ['numpy==2.2.6', 'scipy==1.15.3', 'matplotlib==3.10.9', 'h5py==3.15.1', 'PyYAML==6.0.3', 'unyt==3.0.4']
PINNED_MARKER = Path('/content/.radhydropy_dependencies_pinned_numpy_2_2_6')
if not PINNED_MARKER.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED, '-e', '.[docs]'], cwd=REPO, check=True)
    PINNED_MARKER.touch()
    print('Dependencies installed. Restarting the Colab runtime once to load the pinned NumPy/SciPy binaries.')
    os.kill(os.getpid(), 9)
sys.path.insert(0, str(REPO))
%matplotlib inline
import matplotlib.pyplot as plt
import yaml
from IPython.display import Image, display
plt.rcParams['figure.figsize'] = (9, 5)

In [ ]:
# Copy the smoke configuration and make every output path explicit under /content.
source_dir = REPO / 'example' / 'DynamicStromgrenSpherePhotoheating20pcStellarWind1D'
run_dir = Path('/content/radhydropy-stromgren-wind')
if run_dir.exists():
    shutil.rmtree(run_dir)
run_dir.mkdir()
config = yaml.safe_load((source_dir / 'dynamic_stromgren_sphere_photoheating20pc_stellar_wind1d_smoke.yaml').read_text())
config['par']['output']['directory'] = str(run_dir)
config['par']['simulation']['initial_condition_filename'] = str(run_dir / 'InitialCondition.hdf5')
shutil.copy2(source_dir / 'output_times_smoke.txt', run_dir / 'output_times_smoke.txt')
config['par']['output']['time_list_filename'] = str(run_dir / 'output_times_smoke.txt')
config_file = run_dir / 'stromgren_wind_smoke.yaml'
config_file.write_text(yaml.safe_dump(config))

from example.DynamicStromgrenSpherePhotoheating20pcStellarWind1D import dynamic_stromgren_sphere_photoheating20pc_stellar_wind1d as runner
runner.main(config_file)
print('Completed:', run_dir)

## Inspect the diagnostics

The first figure shows radial profiles of density, velocity, neutral fraction, pressure, and temperature at the final saved time. The ionization-front figure tracks the front radius and compares it with classical reference estimates. The pressure-ratio figure compares wind ram pressure with the thermal pressure of the photoheated gas.

Questions to consider:

- Where is the ionization front, and where is the wind shell?
- Which region is hot but almost fully ionized?
- Is the wind ram pressure larger or smaller than the photoheated-gas pressure at the shell?
- Why should the simulated front differ from the static Strömgren-radius estimate?

In [ ]:
figure_names = [
    'DynamicStromgrenSpherePhotoheating20pcStellarWind1D_C2Ray.jpg',
    'DynamicStromgrenSpherePhotoheating20pcStellarWind1D_C2Ray_IFront.jpg',
    'DynamicStromgrenSpherePhotoheating20pcStellarWind1D_PressureRatio.jpg',
]
for name in figure_names:
    print(name)
    display(Image(filename=str(run_dir / name)))

## From smoke test to production run

The smoke run is deliberately short and low resolution so it fits a teaching session. The production configuration is `dynamic_stromgren_sphere_photoheating20pc_stellar_wind1d.yaml`. It uses 128 active cells, 16 ghost cells, a 1 Myr final time, the same 20 pc domain and physical source parameters, and the full `output_times.txt` schedule. The schedule contains the initial state plus 53 requested output times, so the run produces 54 snapshots.

Do not simply change the smoke run's final time. The production YAML also changes the mesh, timestep controls, hydrogen-source controls, and output schedule. Keep production output separate from the smoke output because the runner removes stale `Output_*.hdf5` files in its configured output directory before starting.

### Safe Colab procedure

1. Finish the smoke run and inspect its figures first.
2. Copy the production YAML and `output_times.txt` into a new directory under `/content`, for example `/content/radhydropy-stromgren-wind-production`.
3. Set `par.output.directory`, `par.simulation.initial_condition_filename`, and `par.output.time_list_filename` to absolute paths inside that directory. This avoids accidentally writing into the cloned repository or using a relative path from the notebook kernel.
4. Import the same maintained runner and call `runner.main(production_config_file)`. The production run is not executed by **Run all** in this teaching notebook; run the optional code only when you are ready for a longer calculation.
5. Save the directory to Google Drive after completion. A Colab runtime reset deletes `/content`.

The production-run template is also provided in the guarded optional cell below; opening the notebook will not launch the long run accidentally:

```python
production_dir = Path('/content/radhydropy-stromgren-wind-production')
production_dir.mkdir(parents=True, exist_ok=True)
production = yaml.safe_load((source_dir / 'dynamic_stromgren_sphere_photoheating20pc_stellar_wind1d.yaml').read_text())
production['par']['output']['directory'] = str(production_dir)
production['par']['simulation']['initial_condition_filename'] = str(production_dir / 'InitialCondition.hdf5')
shutil.copy2(source_dir / 'output_times.txt', production_dir / 'output_times.txt')
production['par']['output']['time_list_filename'] = str(production_dir / 'output_times.txt')
production_config_file = production_dir / 'production.yaml'
production_config_file.write_text(yaml.safe_dump(production))
runner.main(production_config_file)
```

A production run is substantially more expensive than the smoke run because it evolves 1 Myr of coupled hydrodynamics, radiative transfer, chemistry, and thermal source updates. Monitor the output messages and check that the final HDF5 snapshot and C²-Ray figures exist before interpreting the result.

In [ ]:
# Optional production run. Keep False for the normal teaching workflow.
RUN_PRODUCTION = False
if RUN_PRODUCTION:
    production_dir = Path('/content/radhydropy-stromgren-wind-production')
    production_dir.mkdir(parents=True, exist_ok=True)
    production = yaml.safe_load((source_dir / 'dynamic_stromgren_sphere_photoheating20pc_stellar_wind1d.yaml').read_text())
    production['par']['output']['directory'] = str(production_dir)
    production['par']['simulation']['initial_condition_filename'] = str(production_dir / 'InitialCondition.hdf5')
    shutil.copy2(source_dir / 'output_times.txt', production_dir / 'output_times.txt')
    production['par']['output']['time_list_filename'] = str(production_dir / 'output_times.txt')
    production_config_file = production_dir / 'production.yaml'
    production_config_file.write_text(yaml.safe_dump(production))
    runner.main(production_config_file)
    print('Production run completed:', production_dir)

# Optional: save the smoke-run outputs to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# save_dir = Path('/content/drive/MyDrive/RadHydropy/05_stromgren_sphere_with_stellar_wind')
# save_dir.mkdir(parents=True, exist_ok=True)
# shutil.copytree(run_dir, save_dir, dirs_exist_ok=True)
# print('Saved to:', save_dir)